# 第06回：Boltzで複合体を予測する
タンパク質Aと二つの低分子（SAH、チロシン）を入力し、Boltz-2で座標を生成します。Learning ResourcesのBoltz演習と同じ公開入力・固定MSAを使います。

**操作**：Googleアカウントでログイン →「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** →「すべてのセルを実行」。ダウンロードして開き直す必要はありません。編集を残すなら「ドライブにコピーを保存」を使います。

演習20分、採点・提出なし。初回はプログラム・重み・分子辞書のダウンロードがあり、回線とGPUの割り当てにより待ち時間が変わります。GPUを確保できない場合は、計算を開始せず後で再試行してください。

**条件**：1 recycle・20 diffusion steps・1候補・seed 42。操作確認のため反復を減らしています。研究で予測精度を判定する標準条件ではありません。外部MSA検索は行いません。座標生成の反復は分子の実時間の運動ではありません。


In [ ]:
#@title 1. GPUと作業フォルダを確認
from pathlib import Path
import subprocess, sys, time, json, os
from datetime import datetime
started = time.perf_counter()
work = Path('/content/bioinfo06')
work.mkdir(exist_ok=True)
os.chdir(work)
p = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True, capture_output=True)
assert p.returncode == 0, 'T4 GPUを選んでから実行してください。'
print(p.stdout)


## 準備
Colab本体のPythonとは別の環境へBoltzを入れます。再起動は不要です。T4では専用の高速化カーネルを使わず、PyTorchのGPU演算を使います（CPU推論になるわけではありません）。

In [ ]:
#@title 2. Boltz 2.2.1を準備
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'uv', 'py3Dmol==2.5.3'], check=True)
venv = work / '.venv'
py = str(venv / 'bin/python')
if not Path(py).exists():
    subprocess.run(['uv', 'venv', '--python', '3.11', str(venv)], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', py,
                'torch==2.7.1', 'boltz==2.2.1', 'torchmetrics==1.7.4',
                'numpy==1.26.4', 'rdkit==2025.3.3'], check=True)
subprocess.run([py, '-c', "import torch; from importlib.metadata import version; print('Boltz',version('boltz'),'PyTorch',torch.__version__); assert torch.cuda.is_available(); x=torch.ones(4,device='cuda'); print('GPU calculation:',(x+x).cpu().tolist())"], check=True)
print('準備完了。経過秒:', round(time.perf_counter()-started))


## 入力の意味
- **A**：タンパク質のアミノ酸配列。固定した多重配列アラインメント（MSA）を使用。
- **C**：SAH（S-アデノシルホモシステイン）。CCDという化学成分辞書の識別子で指定。
- **E**：チロシン。SMILESという化学構造の文字列表現で指定。立体化学も指定します。

公式の複合体例の重複鎖を一組に減らした入力です。生物学的な仮説を検証するベンチマークではありません。

In [ ]:
#@title 3. 配布したYAMLとMSAを展開する
import base64, zlib, hashlib
inputs = work/'inputs'
inputs.mkdir(exist_ok=True)
yaml_text = "version: 1  # Optional, defaults to 1\nsequences:\n  - protein:\n      id: A\n      sequence: MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ\n      msa: inputs/complex.a3m\n  - ligand:\n      id: C\n      ccd: SAH\n  - ligand:\n      id: E\n      smiles: 'N[C@@H](Cc1ccc(O)cc1)C(=O)O'\n"
msa_bytes = zlib.decompress(base64.b64decode('...'))
(inputs/'complex.yaml').write_text(yaml_text)
(inputs/'complex.a3m').write_bytes(msa_bytes)
print(yaml_text)
print('MSA SHA256:', hashlib.sha256(msa_bytes).hexdigest())


## ノイズから座標を生成
`sampling_steps`は1候補を作る反復数、`diffusion_samples`は生成する候補の数です。`recycling_steps`は条件の特徴表現を更新する反復で、拡散の反復とは別です。学習済みの重みを使い、ここでは学習し直しません。

初回は公式サイトから重みと分子辞書を取得します。ログの進行と最後の「出力確認完了」を確認してください。

In [ ]:
#@title 4. Boltzを実行
sampling_steps = 20 #@param {type:"integer"}
diffusion_samples = 1 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}
assert 2 <= sampling_steps <= 200
assert 1 <= diffusion_samples <= 2
run = work / ('run_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
cmd = [str(venv/'bin/boltz'), 'predict', str(inputs/'complex.yaml'),
       '--out_dir', str(run), '--model', 'boltz2', '--recycling_steps', '1',
       '--sampling_steps', str(sampling_steps), '--diffusion_samples', str(diffusion_samples),
       '--max_parallel_samples', '1', '--seed', str(seed), '--no_kernels',
       '--num_workers', '0', '--write_full_pae', '--write_full_pde']
run.mkdir()
t0 = time.perf_counter()
with (run/'run.log').open('w') as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end=''); log.write(line); log.flush()
    rc = proc.wait()
assert rc == 0, 'ログを確認してください。'
pred = run/'boltz_results_complex/predictions/complex'
cifs = sorted(pred.glob('complex_model_*.cif'))
assert len(cifs) == diffusion_samples, '期待したCIFがありません。ログの失敗数を確認してください。'
for f in cifs:
    assert f.stat().st_size > 1000
meta = {'command':cmd, 'prediction_seconds':time.perf_counter()-t0,
        'total_seconds':time.perf_counter()-started, 'gpu':p.stdout.strip()}
(run/'execution.json').write_text(json.dumps(meta, indent=2))
(run/'requirements.txt').write_text(subprocess.check_output(['uv','pip','freeze','--python',py], text=True))
print('出力確認完了:', len(cifs), '候補。推論・初回データ取得を含む秒:', round(meta['prediction_seconds']))


## 信頼度を読む
BoltzのJSONとNPZではpLDDTは **0〜1** です。以下のグラフは比較しやすいよう100倍します。PAE（Å）は距離そのものではなく、相対配置の予測誤差です。pLDDT・ipTMはモデル自身が予測した信頼度で、実験構造との実測誤差ではありません。結合の強さでもありません。

横軸・縦軸はトークン番号です。タンパク質では残基、低分子では原子がトークンなので、全体を「残基番号」と読まないでください。

In [ ]:
#@title 5. 信頼度の数値とグラフ
import numpy as np
import matplotlib.pyplot as plt
model_index = 0 #@param {type:"integer"}
assert 0 <= model_index < len(cifs)
tag = f'complex_model_{model_index}'
conf = json.loads((pred/f'confidence_{tag}.json').read_text())
for key in ['confidence_score','complex_plddt','ptm','iptm','ligand_iptm','protein_iptm','pair_chains_iptm']:
    if key in conf: print(key, ':', conf[key])
lddt_file = np.load(pred/f'plddt_{tag}.npz')
pae_file = np.load(pred/f'pae_{tag}.npz')
plddt = lddt_file['plddt'].squeeze()
pae = pae_file['pae'].squeeze()
assert np.isfinite(plddt).all() and np.isfinite(pae).all()
assert plddt.min() >= 0 and plddt.max() <= 1.001
fig, ax = plt.subplots(1,2,figsize=(13,4))
ax[0].plot(np.arange(1,len(plddt)+1),100*plddt)
ax[0].set(xlabel='Token index (1-based)',ylabel='Predicted lDDT (0–100)',ylim=(0,100))
im=ax[1].imshow(pae,origin='upper',vmin=0,vmax=30,extent=(.5,pae.shape[1]+.5,pae.shape[0]+.5,.5))
ax[1].set(xlabel='Token index',ylabel='Token index',title='PAE (angstrom)')
fig.colorbar(im,ax=ax[1]);fig.tight_layout()
fig.savefig(run/'confidence.png',dpi=160)
plt.show()


## 立体構造を観察
タンパク質Aを青いリボン、SAH（C）を橙、チロシン（E）を紫のスティックで示します。ドラッグで回転、スクロールで拡大できます。

低分子が置かれた位置と、タンパク質表面との関係を見てください。低分子が近くに見えても、結合が実験で確認されたことにはなりません。

In [ ]:
#@title 6. 複合体の3D表示
import py3Dmol
cif = pred/f'{tag}.cif'
# CIFの座標をBoltz側のgemmiで読み、有限な座標があるか確認する。
check = "import gemmi,sys,math; s=gemmi.read_structure(sys.argv[1]); xyz=[v for m in s for c in m for r in c for a in r for v in (a.pos.x,a.pos.y,a.pos.z)]; assert xyz and all(map(math.isfinite,xyz)); print('Finite atoms:',len(xyz)//3)"
subprocess.run([py,'-c',check,str(cif)],check=True)
view = py3Dmol.view(width=850,height=520)
view.addModel(cif.read_text(),'cif')
view.setStyle({'chain':'A'},{'cartoon':{'color':'steelblue'}})
view.setStyle({'chain':'C'},{'stick':{'color':'orange','radius':.22}})
view.setStyle({'chain':'E'},{'stick':{'color':'purple','radius':.22}})
view.setBackgroundColor('white');view.zoomTo();view.show()


## 結果を残す
ZIPには入力・MSA、予測CIF、信頼度、実行条件、ログ、パッケージ一覧を含めます。Colabのランタイムが消える前に保存してください。重みやインストール環境はZIPへ入れません。

余裕がある場合は`sampling_steps=40`に変え、セル4〜6を再実行します。出力先は実行ごとに分かれます。候補の違いと予測精度の改善は別です。信頼度だけで「正解に近づいた」とは判断しません。

In [ ]:
#@title 7. 結果ZIPを保存
import shutil
from google.colab import files
shutil.copytree(inputs,run/'inputs',dirs_exist_ok=True)
archive = shutil.make_archive(str(run), 'zip', root_dir=run)
print('保存するファイル:', archive)
files.download(archive)


## 出典
- [Boltz公式](https://github.com/jwohlwend/boltz)・[予測の入出力](https://github.com/jwohlwend/boltz/blob/b1ebfc46ecf57f5414e0d1a6f9027bbb122c53bc/docs/prediction.md)
- [Boltz-2論文](https://doi.org/10.1101/2025.06.14.659707)
- 入力：公式examplesを基にしたLearning Resources「Boltzの理論と実装」の複合体例。BoltzコードはMITライセンス。配布MSAと入力の由来も同じ公式例です。
- [Colab_Boltz-2](https://github.com/JKourelis/Colab_Boltz-2)の実行手順を参考に、授業用の固定入力・独立環境・最小限のセル構成で作成しました。
